# 🎯 Day 30 — Mock Interview Practice — **FAANG / Top Product Company Level**

**Week 5 · Day 2 · Sat Nov 21, 2026** · Tags: **10 × [FAANG], 8 × [HARD]**

> Natural step-up from Day 29: Day 29 asked you to implement library-grade algorithms and newer lakehouse surfaces end-to-end. Day 30 goes to the mechanisms interviewers reach for when they want to see *senior* depth: numeric/state-machine SQL (amortization schedules, sequential Elo ratings), data-structure internals you hand-roll (sparse tables, cuckoo hashing), distributed numeric computing (matrix multiply, logistic regression on Spark), scheduling extremes (barrier execution, GPU acceleration), and data-engineering judgment calls (lifecycle tiering, multi-source CDC, junk dimensions, LLM telemetry, Snowflake transaction semantics, and QA practice: synthetic fixtures, replay gates, performance CI).

**18 questions / 8 categories**

| # | Category | Questions |
|---|---|---|
| 1–2 | SQL Advanced | Amortization + underwater LTV [HARD], Elo ratings in recursive SQL [FAANG] |
| 3–4 | Python | Sparse-table RMQ [HARD], Cuckoo hashing [FAANG] |
| 5–6 | PySpark | Blocked matrix multiply [FAANG], Distributed logistic regression [HARD] |
| 7–8 | Spark Optimization | Barrier execution & gang scheduling [HARD], GPU acceleration with RAPIDS [FAANG] |
| 9–11 | ETL & Fault Tolerance | Lifecycle tiering + VACUUM [FAANG], Incremental column backfill [HARD], Multi-source CDC fan-in [FAANG] |
| 12–13 | Data Modeling | Junk dimension [HARD], LLM telemetry star schema [FAANG] |
| 14–15 | Snowflake | No row locks — OCC transaction model [HARD], COPY INTO error modes [FAANG] |
| 16–18 | Monitoring, Logging & QA | Synthetic test data fidelity [FAANG], Consumer-query replay gate [HARD], Query performance regression CI [FAANG] |

**How to practice:** attempt each problem on paper first. Talk track points are printed in each solution — say them out loud. Score yourself in the scorecard at the bottom.

In [0]:
# Day 30 setup — imports + per-run scratch dir (validated pattern from earlier days)
import os, math, uuid, random, time, json, hashlib, statistics, datetime as dt
from collections import Counter
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window

BASE = f"/Volumes/workspace/default/etl_test_volume/d30_{uuid.uuid4().hex[:8]}"
print("scratch dir:", BASE)
print("spark version:", spark.version, "| serverless Spark Connect session")

def file_count(p):
    """Count parquet data files under a Delta table path (no dbutils.fs on serverless)."""
    return len([f for f in os.listdir(p) if f.endswith(".parquet")])

print("setup done")

## Q1 — SQL Advanced — Loan Amortization Schedule + Underwater LTV Detection `[HARD]`

**Scenario (fintech interview, Stripe/SoFi style):** You have a `loans` table (principal, monthly rate, term) and a `homes` table with periodic home-value appraisals. Risk wants two things per loan: (a) a full month-by-month amortization schedule proving principal is fully repaid, and (b) an underwater flag — balance exceeds home value at the appraisal month.

Requirements:
* Generate the schedule with ONE recursive CTE — monthly payment closed form `P·r/(1−(1+r)^−n)`, interest = balance_in × r, principal_part = payment − interest, balance_out = balance_in × (1+r) − payment.
* Watch out: a 360-month mortgage needs `MAX RECURSION LEVEL` in Databricks SQL (default limit is 100).
* Prove correctness: Σ principal_part == original principal (±$0.01), final balance ≈ 0.
* Join to appraisals **at the appraisal month only** (a naive `loan_id`-only join fans out every valuation across every month — the classic trap).
* Flag underwater (LTV > 1.0) and quantify LTV to 4 decimals.

In [0]:
# Solution Q1 — recursive amortization + LTV (validated: principal amortizes exactly, final balance ~0)
loans = spark.createDataFrame([
    ("L1", 200000.0, 0.005, 360),   # 30y @ 6%/yr
    ("L2", 50000.0, 0.004, 60),     # 5y  @ 4.8%/yr
], "loan_id string, principal double, r double, n int")

homes = spark.createDataFrame([
    ("L1", 12, 185000.0), ("L1", 24, 175000.0), ("L1", 48, 210000.0),
    ("L2", 12, 52000.0), ("L2", 36, 44000.0),
], "loan_id string, month int, home_value double")

loans.createOrReplaceTempView("loans"); homes.createOrReplaceTempView("homes")

amort = spark.sql("""
WITH RECURSIVE sched(loan_id, month, payment, balance_in, interest, principal_part, balance_out)
  MAX RECURSION LEVEL 400 AS (
  SELECT l.loan_id, 1, l.principal * l.r / (1 - power(1 + l.r, -l.n)),
         l.principal, l.principal * l.r,
         l.principal * l.r / (1 - power(1 + l.r, -l.n)) - l.principal * l.r,
         l.principal * (1 + l.r) - l.principal * l.r / (1 - power(1 + l.r, -l.n))
  FROM loans l
  UNION ALL
  SELECT s.loan_id, s.month + 1, s.payment, s.balance_out,
         s.balance_out * l.r,
         s.payment - s.balance_out * l.r,
         s.balance_out * (1 + l.r) - s.payment
  FROM sched s JOIN loans l ON s.loan_id = l.loan_id
  WHERE s.month < l.n
)
SELECT * FROM sched
""")
amort.cache()

# correctness: principal fully repaid; final month's balance ~0 (row_number, NOT max()!)
chk = amort.groupBy("loan_id").agg(
    F.round(F.sum("principal_part"), 4).alias("sum_principal"),
    F.round(F.sum("interest"), 4).alias("total_interest")
).toPandas()
w = Window.partitionBy("loan_id").orderBy(F.col("month").desc())
fb = (amort.withColumn("rn", F.row_number().over(w)).filter("rn = 1")
      .select("loan_id", F.round("balance_out", 6).alias("final_balance"))).toPandas()
print(chk.to_string(index=False)); print(fb.to_string(index=False))

# LTV at appraisal months — join on (loan_id, month), NOT loan_id alone
ltv_pd = (amort.join(homes, ["loan_id", "month"])
          .withColumn("ltv", F.round(F.col("balance_out") / F.col("home_value"), 4))
          .select("loan_id", "month", "home_value", "ltv",
                  (F.col("balance_out") > F.col("home_value")).alias("underwater"))
          .orderBy("loan_id", "month").toPandas())
print(ltv_pd.to_string(index=False))

assert abs(float(chk.loc[chk.loan_id == "L1", "sum_principal"].iloc[0]) - 200000.0) < 0.01
assert abs(float(fb.loc[fb.loan_id == "L1", "final_balance"].iloc[0])) < 0.01
assert abs(float(chk.loc[chk.loan_id == "L2", "sum_principal"].iloc[0]) - 50000.0) < 0.01
assert len(ltv_pd) == 5
assert ltv_pd.loc[(ltv_pd.loan_id == "L1") & (ltv_pd.month == 24), "underwater"].iloc[0] == True
assert ltv_pd.loc[(ltv_pd.loan_id == "L1") & (ltv_pd.month == 48), "underwater"].iloc[0] == False
assert abs(float(ltv_pd.loc[(ltv_pd.loan_id == "L2") & (ltv_pd.month == 36), "ltv"].iloc[0]) - 0.4874) < 0.01
print("Q1 OK: principal amortized exactly, final balance ~0, L1 underwater at month 24 (1.1139), recovered by 48 (0.9011)")
print("\nTalk track:\n* MAX RECURSION LEVEL is the Databricks-specific knob; without it a 360-month loan dies at depth 100.")
print("* Final balance must come from the LAST row (row_number desc) — max(balance_out) returns month 1, a classic bug.")
print("* Underwater risk is only meaningful at appraisal points; joining on loan_id alone fans out every valuation x every month.")

## Q2 — SQL Advanced — Sequential Elo Rating Updates in Pure SQL `[FAANG]`

**Scenario (Matchmaking-team interview, ELO at scale):** You have a `matches` stream (match_no, winner, loser). Ratings start at 1500 and each match must be applied **in order**: `R' = R + K·(S − E)`, `E = 1/(1+10^((R_opp − R_self)/400))`. This is inherently sequential state — the hard ask is doing it in one SQL statement, no loops.

Requirements:
* Use a recursive CTE whose carried state is a **MAP<player, rating>** — one row per step, not one row per player-state.
* Per match: `element_at(map, key)` reads both ratings; `map_filter` strips the two players; `map_concat` re-adds updated entries.
* Join the recursive state to `matches ON match_no = step + 1` — the recursive term may reference itself only once, but joining a non-recursive table is legal.
* Verify against a Python sequential reference to 1e-4; report the final ranking.
* Say why this is O(matches) recurrences and when it beats shuffling the job to a driver loop.

In [0]:
# Solution Q2 — Elo via recursive CTE carrying a map (validated == Python sequential reference to 1e-4)
matches = spark.createDataFrame([
    (1, "alice", "bob"), (2, "bob", "carol"), (3, "carol", "alice"), (4, "alice", "bob"),
], "match_no int, winner string, loser string")
matches.createOrReplaceTempView("matches")

elo_row = spark.sql("""
WITH RECURSIVE st(step, ratings) MAX RECURSION LEVEL 50 AS (
  SELECT 0, map_from_entries(collect_list(struct(name, cast(1500.0 as double))))
  FROM (SELECT winner AS name FROM matches UNION SELECT loser AS name FROM matches)
  UNION ALL
  SELECT s.step + 1,
         map_concat(
           map_filter(s.ratings, (k, v) -> k NOT IN (m.winner, m.loser)),
           map(m.winner,
               element_at(s.ratings, m.winner) + 20 * (1 - 1/(1 + power(10, (element_at(s.ratings, m.loser) - element_at(s.ratings, m.winner))/400.0)))),
           map(m.loser,
               element_at(s.ratings, m.loser) + 20 * (0 - 1/(1 + power(10, (element_at(s.ratings, m.winner) - element_at(s.ratings, m.loser))/400.0))))
         )
  FROM st s JOIN matches m ON m.match_no = s.step + 1
)
SELECT step, ratings FROM st ORDER BY step DESC LIMIT 1
""")
r = elo_row.collect()[0]
sql_ratings = {k: round(v, 6) for k, v in r["ratings"].items()}
print(f"SQL final ratings after {r['step']} matches:", sql_ratings)

def elo_ref(ms, K=20, R0=1500.0):
    ratings = {}
    for _, w, l in ms:
        ratings.setdefault(w, R0); ratings.setdefault(l, R0)
        ew = 1 / (1 + 10 ** ((ratings[l] - ratings[w]) / 400.0))
        el = 1 / (1 + 10 ** ((ratings[w] - ratings[l]) / 400.0))
        ratings[w] += K * (1 - ew)
        ratings[l] += K * (0 - el)
    return ratings

ref = {k: round(v, 6) for k, v in elo_ref(matches.collect()).items()}
print("Python reference:      ", ref)
assert set(sql_ratings) == set(ref) and len(sql_ratings) == 3
for k in ref:
    assert abs(sql_ratings[k] - ref[k]) < 1e-4, (k, sql_ratings[k], ref[k])
rank = sorted(sql_ratings.items(), key=lambda kv: -kv[1])
print("ranking:", rank)
assert rank[0][0] == "alice"
print("Q2 OK: SQL map-state Elo == Python sequential reference to 1e-4; alice #1 after beating bob twice")
print("\nTalk track:\n* The whole trick is state-as-VALUE: one map row per step. Splitting state across rows breaks the single self-reference rule.")
print("* element_at/map_filter/map_concat are the map-manipulation trio worth memorizing for FAANG SQL rounds.")
print("* Cost: N recurrences, each rebuilding an O(players) map — fine to thousands of players; for millions, do it on the driver or in Spark Structured Streaming with mapGroupsWithState.")

## Q3 — Python — Range Minimum Query: Sparse Table with O(1) Queries `[HARD]`

**Scenario (systems-interview classic):** A monitoring service answers "what was the minimum p99 latency between positions l and r?" millions of times against an immutable array. Segment trees give O(log n); the senior answer for **static** arrays is a **sparse table**: O(n log n) build, **O(1) per query**.

Requirements:
* Build `st[k][i] = min over [i, i + 2^k − 1]` from `st[k−1]` blocks — no fancy libraries.
* Query(l, r) covers [l, r] with two possibly-overlapping precomputed blocks of length 2^k where `k = floor(log2(r−l+1))` — overlap is fine because min is idempotent.
* Validate on a 500-element random array against naive `min(arr[l:r+1])` on 500 random queries, plus the length-1 and whole-array edges.
* Talk track: why idempotency (min/max/AND) is required and why this dies for non-idempotent aggregates like sum.

In [0]:
# Solution Q3 — sparse table RMQ (validated: 500 random queries == naive, incl. edges)
def build_sparse_table(arr):
    n = len(arr)
    k = n.bit_length()
    st = [[0] * n for _ in range(k)]
    st[0] = arr[:]
    for j in range(1, k):
        step = 1 << (j - 1)
        for i in range(0, n - (1 << j) + 1):
            st[j][i] = min(st[j - 1][i], st[j - 1][i + step])
    return st

def query_sparse_table(st, l, r):
    j = (r - l + 1).bit_length() - 1
    return min(st[j][l], st[j][r - (1 << j) + 1])

random.seed(30)
arr = [random.randint(-100, 100) for _ in range(500)]
st = build_sparse_table(arr)
for _ in range(500):
    l = random.randrange(500); r = l + random.randrange(500 - l)
    assert query_sparse_table(st, l, r) == min(arr[l:r + 1]), (l, r)
assert query_sparse_table(st, 0, 0) == arr[0]
assert query_sparse_table(st, 0, 499) == min(arr)
n = 500
print(f"table levels: {n.bit_length()} (n=500), memory = n·log n ints = {n * n.bit_length()}")
print("Q3 OK: 500 random queries == naive min, incl. len-1 and full-array edges")
print("\nTalk track:")
print("* Build is n·log n; every query is exactly 2 array lookups + 1 min -> true O(1), no tree walk.")
print("* The trick: two blocks of length 2^k starting at l and ending at r may overlap — idempotent min makes overlap harmless.")
print("* Non-idempotent aggregates (sum) can't overlap -> use prefix sums (static) or a segment tree/Fenwick (mutable).")
print("* RMQ over immutable arrays is also the engine under LCA in suffix-tree work — worth naming in interviews.")

## Q4 — Python — Cuckoo Hashing: O(1) Worst-Case Lookups `[FAANG]`

**Scenario (cache-infra interview):** Your LRU cache's worst-case lookup is O(n) per bucket under adversarial keys. **Cuckoo hashing** guarantees each key lives in exactly one of TWO slots (one per hash function) — lookup and delete check at most two places, worst-case O(1). The price is insert-time churn: an evicted key hops between tables, and cycles must be detected.

Requirements:
* Implement insert with hop-alternation between t1/h1 and t2/h2, cycle detection via a hop limit.
* On cycle: production behavior is rehash into a bigger table with new hash salt (bounded retries), else fail gracefully — never corrupt state or lose an existing key.
* Support in-place update, delete, lookup; verify lookups after churn.
* Demonstrate graceful rejection on a saturated tiny table.

In [0]:
# Solution Q4 — cuckoo hashing with bounded rehash (validated: 14 keys churn-tested; saturated tiny table rejects gracefully)
class CuckooHash:
    def __init__(self, capacity=16, max_hops=40, max_rehash=3):
        self.m = capacity
        self.max_hops = max_hops
        self.max_rehash = max_rehash
        self.salt = 1
        self.t1 = [None] * self.m
        self.t2 = [None] * self.m
        self.size = 0

    def h1(self, key): return hash(("t1", key, self.salt)) % self.m
    def h2(self, key): return hash(("t2", key, self.salt)) % self.m

    def _insert_no_rehash(self, k, v):
        for _ in range(self.max_hops):
            i = self.h1(k)
            if self.t1[i] is None:
                self.t1[i] = (k, v); self.size += 1; return True
            k2, v2 = self.t1[i]; self.t1[i] = (k, v); k, v = k2, v2
            i = self.h2(k)
            if self.t2[i] is None:
                self.t2[i] = (k, v); self.size += 1; return True
            k2, v2 = self.t2[i]; self.t2[i] = (k, v); k, v = k2, v2
        return False

    def _rehash(self):
        pairs = [p for p in self.t1 + self.t2 if p is not None]
        self.m *= 2
        self.salt += 1
        self.t1 = [None] * self.m
        self.t2 = [None] * self.m
        self.size = 0
        for k, v in pairs:
            if not self._insert_no_rehash(k, v):
                return False
        return True

    def insert(self, key, val):
        if self.contains(key):
            self.t1[self.h1(key)] = (key, val); return True          # in-place update
        if self._insert_no_rehash(key, val):
            return True
        for _ in range(self.max_rehash):                              # cycle -> grow + rehash
            if self._rehash() and self._insert_no_rehash(key, val):
                return True
        return False

    def contains(self, key):
        c = self.t1[self.h1(key)]
        if c is not None and c[0] == key: return True
        c = self.t2[self.h2(key)]
        if c is not None and c[0] == key: return True
        return False

    def lookup(self, key):
        c = self.t1[self.h1(key)]
        if c is not None and c[0] == key: return c[1]
        c = self.t2[self.h2(key)]
        if c is not None and c[0] == key: return c[1]
        return None

    def delete(self, key):
        for t, h in ((self.t1, self.h1), (self.t2, self.h2)):
            i = h(key)
            if t[i] is not None and t[i][0] == key:
                t[i] = None; self.size -= 1; return True
        return False

ch = CuckooHash(capacity=16)
for i in range(14):
    assert ch.insert(f"key{i}", i * 10), i
assert all(ch.lookup(f"key{i}") == i * 10 for i in range(14))
assert ch.lookup("missing") is None
assert ch.delete("key5") and ch.lookup("key5") is None and not ch.delete("key5")
assert ch.insert("key5", 55) and ch.lookup("key5") == 55
assert ch.insert("key0", 999) and ch.lookup("key0") == 999      # in-place update
print(f"capacity after inserts: m={ch.m}, size={ch.size} (load {ch.size/(2*ch.m):.2f})")

tiny = CuckooHash(capacity=2, max_hops=10, max_rehash=0)
fails = sum(1 for i in range(30) if not tiny.insert(f"k{i}", i))
print(f"tiny m=2 no-rehash table: {fails} of 30 inserts rejected (graceful cycle detect)")
assert fails > 0
print("Q4 OK: 14 keys insert/lookup/update/delete/reinsert, bounded-rehash growth, graceful saturation rejection")
print("\nTalk track:")
print("* Lookup/delete touch at most 2 slots -> worst-case O(1), the whole point. Insert amortizes to O(1) below ~0.5 load.")
print("* Eviction chains form the name: a new key kicks out the occupant, which kicks out another, like a cuckoo egg.")
print("* Cycle = eviction returns to a visited slot. Detect with a hop cap; recover by rehash (bigger tables, new salt).")
print("* vs chaining: no per-bucket lists, better cache locality; vs linear probing: no clustering, but higher insert cost.")

## Q5 — PySpark — Distributed Blocked Matrix Multiply `[FAANG]`

**Scenario (Spark-numerics interview):** "Multiply a 12k×9k by a 9k×6k matrix on Spark — no collect(), no numpy on the whole thing." This is the classic join-based MM, and the block bookkeeping is what interviewers probe.

Requirements:
* Explode A into cells `(bi, bj, i, j, v)` with block indices at block size 3 — same for B with `(bj, bk, i, j, v)` (B's row block index IS the shared middle index).
* Join on the middle dimension (`a_bj = b_bi` AND `a.j = b.i`), multiply pairwise, then SUM per output cell `(bi, bk, i, j)`.
* Block-local aggregation: every 3×3 output block must hold exactly 9 cells — prove it.
* Verify the full result against `numpy A@B` (int-valued matrices make the comparison exact to 1e-9).
* Talk track: where the shuffle happens, why one giant skew-less join beats nested-loop per block at scale, and the comms/compute ratio that makes Spark MM lose to BLAS until matrices are huge.

In [0]:
# Solution Q5 — join-based MM with block bookkeeping (validated == numpy A@B at 1e-9; blocks all 9 cells)
random.seed(7)
A = np.array([[random.randint(-5, 5) for _ in range(9)] for _ in range(12)], dtype=float)
B = np.array([[random.randint(-5, 5) for _ in range(6)] for _ in range(9)], dtype=float)
BS = 3  # block size

def to_cell_df(M, prefix):
    n, k = M.shape
    return spark.createDataFrame(
        [(i // BS, j // BS, i, j, float(M[i, j])) for i in range(n) for j in range(k)],
        f"{prefix}bi int, {prefix}bj int, i int, j int, v double")

dfA = to_cell_df(A, "a_").alias("A")   # a_bi, a_bj, i, j, v
dfB = to_cell_df(B, "b_").alias("B")   # b_bi, b_bj, i, j, v

mm = (dfA.join(dfB, (F.col("A.a_bj") == F.col("B.b_bi")) & (F.col("A.j") == F.col("B.i")))
      .select(F.col("A.a_bi").alias("bi"), F.col("B.b_bj").alias("bj"),
              F.col("A.i").alias("i"), F.col("B.j").alias("j"),
              (F.col("A.v") * F.col("B.v")).alias("prod"))
      .groupBy("bi", "bj", "i", "j").agg(F.sum("prod").alias("v")))

res = np.zeros((12, 6))
for row in mm.collect():
    res[row["i"], row["j"]] = row["v"]
expected = A @ B
assert np.allclose(res, expected, atol=1e-9), np.abs(res - expected).max()

blocks = (mm.groupBy("bi", "bj").agg(F.count("*").alias("cells"), F.sum("v").alias("block_sum"))
          .orderBy("bi", "bj"))
bl = blocks.toPandas()
assert set(bl["cells"]) == {9}                                   # every 3x3 block complete
assert abs(bl["block_sum"].sum() - float(expected.sum())) < 1e-6
print(f"block grid {bl.bi.nunique()}x{bl.bj.nunique()}, every block 9 cells, block sums == A@B total {float(expected.sum()):.1f}")
print("Q5 OK: distributed cell-level MM with block bookkeeping == numpy A@B exactly")
print("\nTalk track:")
print("* Shuffle: the join on the middle dimension IS the (i,k) partial-product exchange; groupBy(bi,bk,i,j) reduces locally first.")
print("* Cost: A.cells x B.cells/k-dim multiplications; block indices let AQE/bucketing co-partition matching blocks.")
print("* In production you'd aggregate at BLOCK level (per-block partial matrices) to shrink shuffle volume; cell-level keeps the demo exact.")
print("* Spark only beats single-node BLAS when data can't fit RAM — comms dominates otherwise.")

## Q6 — PySpark — Distributed Logistic Regression via Per-Partition Gradients `[HARD]`

**Scenario (ML-infra interview):** Train logistic regression on 10B rows without collecting data. The pattern behind every large-scale ML engine (and the ancestor of parameter-server training): **broadcast weights, compute per-partition gradient sums, reduce on the driver, iterate.**

Requirements:
* Generate synthetic data from known weights (0.8, 1.2, −0.7); repartition to 4 partitions.
* `mapInPandas` yields per-batch gradient partials g0 (intercept), g1, g2 — flat 1-D `err = p − y`, NEVER a column vector (a (n,1) x (n,) broadcast silently multiplies your gradient by n!).
* Each iteration: distributed gradient must equal the single-node full-batch gradient to 1e-8 — assert it EVERY iteration.
* 200 iterations of gradient descent (lr=0.5), report accuracy > 0.72 and correct weight signs.
* Talk track: why mapInPandas beats a UDF per-row, and how this becomes all-reduce/MPI at scale.

In [0]:
# Solution Q6 — broadcast-weights mapInPandas training loop (validated: grad == full-batch ref 1e-8 every iter; acc 0.73)
random.seed(8)
n, d = 2000, 3   # 2 features + intercept
wtrue = np.array([0.8, 1.2, -0.7])
Xgen = np.random.randn(n, 2)
p = 1 / (1 + np.exp(-(Xgen @ wtrue[1:] + wtrue[0])))
y = (np.random.rand(n) < p).astype(float)

train = spark.createDataFrame(
    [(float(y[i]), float(Xgen[i, 0]), float(Xgen[i, 1])) for i in range(n)],
    "label double, x1 double, x2 double").repartition(4, F.monotonically_increasing_id())

def make_grad_udf(w):
    w = np.array(w, dtype=float)
    def g(batch_iter):
        for pdf in batch_iter:
            Xb = np.column_stack([np.ones(len(pdf)), pdf.x1.to_numpy(), pdf.x2.to_numpy()])
            yb = pdf.label.to_numpy()
            pr = 1 / (1 + np.exp(-(Xb @ w)))
            err = pr - yb                                  # FLAT 1-D — no (n,1)x(n,) broadcast blowup
            yield pd.DataFrame({"g0": [float((err * Xb[:, 0]).sum())],
                                "g1": [float((err * Xb[:, 1]).sum())],
                                "g2": [float((err * Xb[:, 2]).sum())]})
    return g

Xfull = np.column_stack([np.ones(n), Xgen])
def full_batch_grad(w):
    pr = 1 / (1 + np.exp(-(Xfull @ w)))
    return Xfull.T @ (pr - y)

w = np.zeros(3); lr = 0.5
for it in range(200):
    parts = train.mapInPandas(make_grad_udf(w), "g0 double, g1 double, g2 double").toPandas()
    g_dist = parts[["g0", "g1", "g2"]].sum().to_numpy()
    g_ref = full_batch_grad(w)
    assert np.allclose(g_dist, g_ref, atol=1e-8), (it, g_dist, g_ref)   # distributed == single-node
    w -= lr * g_ref / n

acc = float(((1 / (1 + np.exp(-(Xfull @ w))) > 0.5) == y).mean())
print(f"trained weights: {np.round(w, 4)} | true: {wtrue} | train accuracy: {acc:.3f}")
assert acc > 0.72 and np.sign(w[1]) == np.sign(wtrue[1]) and np.sign(w[2]) == np.sign(wtrue[2])
print("Q6 OK: per-partition gradient sums == full-batch reference every iteration; converges; signs match truth")
print("\nTalk track:")
print("* mapInPandas gets whole batches -> numpy vector math per batch, 1 row out per batch: tiny shuffle, no per-row Python.")
print("* The killer bug: err as (n,1) column vector makes err*Xb[:,0] an (n,n) matrix — silent, wrong by ~n. Found the hard way in validation.")
print("* At 10B rows this loop becomes: all-reduce the partials across executors (Spark barrier mode or MPI-style) and iterate on parameter servers.")

## Q7 — Spark Optimization — Barrier Execution Mode & Gang Scheduling `[HARD]`

**Scenario (distributed-training interview):** You run iterative training / MPI-style algorithms where every iteration needs ALL executors alive simultaneously. One straggler restarts its task alone in default FAIR/FIFO mode — fine for ETL, fatal for synchronization steps that exchange state.

Requirements (talk-track + quantitative sim):
* Explain barrier stages: `df.barrier()` marks a stage all-or-nothing — Spark launches ALL barrier tasks together, tracks them via `BarrierTaskInfo`, and if ANY task fails, the WHOLE stage is retried (no partial progress).
* The constraints interviewers test: barrier stages reject speculative execution, require equal `spark.scheduler.barrier.maxConcurrentTasksCheck` vs parallelism, one partition per task slot, and coexist poorly with AQE coalescing.
* Compare with default mode: a task failing mid-stage loses only that task (checkpointed RDD lineage), but no gang guarantee — stragglers cause skew-style stalls.
* Simulate: 10 tasks, per-task failure prob 2%, task time 60s — compute expected stage-completion time under (a) default: failed tasks restart individually; (b) barrier: any failure restarts the whole gang. Say when each wins.

In [0]:
# Solution Q7 — talk track + expected-time simulation (deterministic math, asserted)
print("""
Talk track — barrier execution mode:
* df.barrier() marks the NEXT stage all-or-nothing. Spark requests all barrier tasks at once (gang scheduling) and
  tracks them via BarrierTaskInfo; barrierc() style sync points (context.barrier()) let the stage itself synchronize.
* Failure semantics flip: default mode loses ONE task and retries it alone; barrier mode aborts and retries the
  ENTIRE stage — every task re-runs from the stage boundary. That's the price of a gang guarantee.
* Hard constraints: no speculative execution, no salting/skew splits inside the stage, parallelism must be
  reproducible (maxConcurrentTasks check interval), each partition = one task = one slot, and AQE coalescing
  can't rebalance a barrier stage — size partitions before.
* Where it's used: distributed training iterations, MPI-style collectives (all-reduce), graph algorithms with
  superstep sync (Pregel-style), exactly-co-partitioned state exchange.
* Configs: spark.scheduler.barrier.maxConcurrentTasksCheck.[maxFailures,interval]
""")

# --- expected stage-completion time: default vs barrier, deterministic simulation ---
random.seed(17)
N_TASKS, TASK_S, P_FAIL, TRIALS = 10, 60.0, 0.05, 2000

def sim_default():
    # each task retries independently until it succeeds; stage done when all succeed
    retries = 0
    for _ in range(N_TASKS):
        while random.random() < P_FAIL:   # this task failed once -> restart it
            retries += 1
    return TASK_S * (1 + retries)         # failed task restarts add full task time

def sim_barrier():
    # any single task failure restarts the WHOLE stage (all N tasks redo their work)
    stage_time = TASK_S
    while True:
        failures = sum(1 for _ in range(N_TASKS) if random.random() < P_FAIL)
        if failures == 0:
            return stage_time
        stage_time += TASK_S               # full gang redo

d = [sim_default() for _ in range(TRIALS)]
b = [sim_barrier() for _ in range(TRIALS)]
import statistics as st
mean_d, mean_b = st.mean(d), st.mean(b)
print(f"mean stage time @ p_fail=5%: default={mean_d:.1f}s  barrier={mean_b:.1f}s  (n={TRIALS} trials each)")
print(f"medians (both usually one clean pass): default={st.median(d):.0f}s  barrier={st.median(b):.0f}s")
assert mean_d > TASK_S                    # failures cost even in default mode
assert mean_b > mean_d + 5                # barrier pays the gang-restart tax on EXPECTED time

# flip: straggler-prone cluster WITHOUT failures — barrier's gang placement prevents stragglers
print("\nStraggler math: if tasks are placed by a gang scheduler onto homogeneous slots, per-task\nvariance collapses; barrier wins when stragglers (not crashes) dominate — e.g. p99 task = 2x median\nin default mode vs fixed slots in barrier mode.")
print("Q7 OK: at p_fail=5%% default wins on expected stage time (%.1fs vs %.1fs); barrier wins on straggler-prone or co-partitioned workloads where per-task progress without the gang is useless." % (mean_d, mean_b))

## Q8 — Spark Optimization — GPU-Accelerated Spark with RAPIDS `[FAANG]`

**Scenario (2025-trending: GPU data engineering):** Your feature pipeline (wide projections, string ops, numeric transforms) is CPU-bound; the org bought A100s. The **RAPIDS Spark plugin** runs cuDF columnar kernels inside the JVM with ZERO code change — but data must get TO the GPU. Interviewers test whether you can reason about when the GPU actually wins.

Requirements (talk-track + transfer math):
* Explain the architecture: plugin replaces physical plan operators with GPU versions (`spark.rapids.sql.enabled=true`); unsupported ops fall back to CPU (watch the split — GPU→CPU→GPU round trips kill you).
* Quantify the transfer bottleneck: PCIe Gen4 ~25 GB/s, batch of n rows × 16 bytes payload — compute per-batch transfer time and the GPU compute time needed to amortize it.
* Derive break-even: `speedup = cpu_time / (transfer_time + gpu_time)`; solve for the minimum batch size where GPU wins if the GPU kernel is 20× faster.
* Say when GPU wins (wide numeric/string ops, big batches, ML preprocessing, shuffle-light) and loses (tiny batches, UDF-heavy Python, shuffle-dominated, sparse ops).

In [0]:
# Solution Q8 — talk track + break-even math (deterministic, asserted)
print("""
Talk track — RAPIDS on Spark:
* The plugin (nvidia/spark-rapids) swaps Catalyst physical operators for cuDF GPU kernels — same DataFrame code,
  spark.rapids.sql.enabled=true on a GPU cluster. Columnar cuDF runs INSIDE the JVM (no Python round trip).
* Unsupported expressions fall back to CPU. The silent killer is plan splits: data bounces GPU->CPU->GPU,
  paying the PCIe toll each way. Read the plan (EXPLAIN) for 'GpuProject'/'GpuFilter' vs CPU islands.
* Comms hierarchy: PCIe Gen4 ~25 GB/s host<->GPU; NVLink ~450-900 GB/s GPU<->GPU; a shuffle is host-side
  unless RAPIDS shuffle (UCX) is enabled. The first question for any GPU plan: how many bytes cross PCIe per useful kernel?
* Databricks: GPU node types (e.g. g5.2xlarge A10G) + RAPIDS runtime, or Spark RAPIDS on K8s; Photon is CPU-vectorized
  and often already good enough for ETL — GPU pays off for numeric-heavy transforms and ML feature pipelines.
""")

# --- transfer + amortization math ---
PCIE_GB_S = 25.0        # PCIe Gen4, one direction, realistic
ROW_BYTES = 16.0        # e.g. 8 cols of int/double payload
gpu_speedup = 20.0      # kernel speedup vs CPU

def transfer_s(batch_rows):
    return (batch_rows * ROW_BYTES) / (PCIE_GB_S * 1e9)

for rows in (1_000, 100_000, 10_000_000):
    t = transfer_s(rows)
    print(f"batch {rows:>10,} rows = {rows * ROW_BYTES / 1e9:6.2f} GB -> PCIe transfer {t * 1000:8.2f} ms")

# break-even: cpu_time = transfer + gpu_time, gpu_time = cpu_time / speedup
#   cpu_time * (1 - 1/speedup) = transfer  ->  cpu_time = transfer / (1 - 1/speedup)
rows = 10_000_000
t = transfer_s(rows)
cpu_time_be = t / (1 - 1 / gpu_speedup)
print(f"\nbreak-even at 10M-row batches: transfer {t*1000:.1f} ms -> GPU wins iff CPU time per batch > {cpu_time_be*1000:.0f} ms (kernel 20x faster)")

# 1M-row batches where the CPU op is trivial (0.5ms): does the GPU still win?
rows_small = 1_000_000
t_small = transfer_s(rows_small)
cpu_small = 0.0005   # 0.5 ms trivial CPU op — moving data costs more than the op itself
gpu_total = t_small + cpu_small / gpu_speedup
print(f"1M-row batch, CPU kernel 0.5 ms: GPU total = {(t_small + cpu_small/gpu_speedup)*1000:.2f} ms vs CPU 0.50 ms -> {'GPU wins' if gpu_total < cpu_small else 'CPU wins'}")
assert gpu_total > cpu_small             # transfer dominates: CPU wins on small fast batches

# 100M-row batch with 30s of CPU work: GPU wins?
rows_big, cpu_big = 100_000_000, 30.0
t_big = transfer_s(rows_big)
gpu_total_big = t_big + cpu_big / gpu_speedup
print(f"100M-row batch, CPU kernel 30 s: GPU total = {gpu_total_big:.2f} s vs CPU 30.00 s -> {'GPU wins' if gpu_total_big < cpu_big else 'CPU wins'}")
assert gpu_total_big < cpu_big
print("Q8 OK: math shows GPU wins only when (kernel work x speedup) exceeds the PCIe toll — tiny batches never amortize")

## Q9 — ETL — Data Lifecycle Tiering: Hot/Warm/Cold + VACUUM + Cost Math `[FAANG]`

**Scenario (cost-interview):** A 5 TB orders table grows 40 GB/day; 30 days of history serve 99% of queries, older data is read ~monthly for audits. Storage is your #2 line item. Design the lifecycle policy and prove the math.

Requirements:
* LIVE: show that an UPDATE leaves orphaned files under copy-on-write (file count grows) and that a default-retention VACUUM reclaims NOTHING while files are younger than 7 days — retention gates reclamation (run `VACUUM ... RETAIN 0 HOURS` only in a maintenance window with no concurrent writers; it permanently deletes time-travel history beyond retention — destructive).
* Tiering: keep 90 days hot (cloud object storage standard ~$0.023/GB-mo), archive older partitions to a cold tier (~$0.004/GB-mo + ~$0.01/GB retrieval) as Parquet snapshots with a manifest.
* Derive the break-even: how many full re-reads per month make cold storage MORE expensive than hot? (Answer: ~1.9 — audit the math.)
* Talk track: deletedFileRetentionDuration vs logRetentionDuration, restore latency expectations, and why reads-from-cold must go through the manifest, not the Delta log.

In [0]:
# Solution Q9 — live orphan-file demo + tiering cost math (validated)
q9 = f"{BASE}_q9_orders"
rows = [(i, f"2026-01-{(i % 28) + 1:02d}", float(i * 3)) for i in range(1, 61)]
(spark.createDataFrame(rows, "id int, order_date string, amount double")
 .write.mode("overwrite").format("delta").save(q9))

before = file_count(q9)
spark.sql(f"UPDATE delta.`{q9}` SET amount = amount * 1.1 WHERE id <= 20")   # copy-on-write: old file orphaned
after_upd = file_count(q9)

spark.sql(f"VACUUM delta.`{q9}`")        # default 168h retention — orphans are too young, nothing reclaimed
# spark.sql(f"VACUUM delta.`{q9}` RETAIN 0 HOURS")   # destructive: only in a maintenance window, no writers — docs only

hist = spark.sql(f"DESCRIBE HISTORY delta.`{q9}`").select("version", "operation").toPandas()
print(hist.to_string(index=False))
print(f"files: initial={before}, after UPDATE={after_upd}")
assert after_upd > before, "UPDATE must orphan files (copy-on-write)"

# --- tiering cost math ---
hot_pergb_mo, archive_pergb_mo, retrieval_pergb = 0.023, 0.004, 0.01
gb = 5000.0
hot_cost = gb * hot_pergb_mo
archive_cost = gb * archive_pergb_mo
print(f"5TB/mo: hot ${hot_cost:.0f} vs archive ${archive_cost:.0f} -> saves ${hot_cost - archive_cost:.0f}/mo")
x = (hot_pergb_mo - archive_pergb_mo) / retrieval_pergb   # re-reads that erase the savings
print(f"break-even: archive cheaper unless you re-read > {x:.1f}x the dataset per month")
assert abs(x - 1.9) < 0.05
print("Q9 OK: UPDATE orphans files live; retention gates VACUUM; tiering break-even = 1.9 full re-reads/month")
print("\nTalk track:")
print("* VACUUM removes files older than deletedFileRetentionDuration (default 7 days); the LOG is pruned by logRetentionDuration (30 days). VACUUM 0 HOURS is irreversible — kills time travel; window it.")
print("* Cold tier = partitioned Parquet snapshots + manifest table (partition, path, rows, bytes, archived_at). Reads join the manifest; restore = re-ingest, not Delta time travel.")
print("* Audit frequency decides the tier: monthly audits (0.25 re-reads/mo) << 1.9 break-even -> cold wins; ad-hoc BI on old data -> keep hot or cache.")

## Q10 — ETL — Incremental Column Backfill with a Progress Ledger `[HARD]`

**Scenario:** Marketing asks for a `value_bucket` on a 50M-row events table. `ALTER TABLE ADD COLUMN` is metadata-only and instant — but backfilling 50M rows in one UPDATE is a multi-hour lock-holding monster that could die at 90%.

Requirements:
* Prove ALTER ADD COLUMN does NOT rewrite files (file count unchanged).
* Backfill in batches of 10 ids via `MERGE ... WHEN MATCHED AND value_bucket IS NULL THEN UPDATE` — each batch scoped `WHERE id BETWEEN lo AND hi` so a failure loses one batch, not the run.
* Maintain a **progress ledger** Delta table of completed batch ids; a re-run must skip completed batches (idempotent) — prove first run = 5 batches, re-run = 0.
* Prove zero NULLs remain, bucket boundaries correct, and new inserts carrying the column work untouched.

In [0]:
# Solution Q10 — metadata-only ALTER + ledgered batch MERGE (validated: 5 batches, re-run 0, no nulls)
q10 = f"{BASE}_q10_events"
rows = [(i, f"user{i % 7}", float(i)) for i in range(1, 51)]
(spark.createDataFrame(rows, "id int, user string, value double")
 .write.mode("overwrite").format("delta").save(q10))
files_before = file_count(q10)

spark.sql(f"ALTER TABLE delta.`{q10}` ADD COLUMN value_bucket STRING")   # metadata-only
assert file_count(q10) == files_before, "ALTER ADD COLUMN must NOT rewrite files"
assert "value_bucket" in [f.name for f in spark.read.format("delta").load(q10).schema.fields]

ledger = f"{BASE}_q10_ledger"
(spark.createDataFrame([(-1,)], "batch_id int").write.mode("overwrite").format("delta").save(ledger))

def run_backfill(batch_size=10):
    done = {r["batch_id"] for r in spark.sql(f"SELECT batch_id FROM delta.`{ledger}`").collect() if r["batch_id"] > 0}
    total = spark.sql(f"SELECT max(id) FROM delta.`{q10}`").collect()[0][0]
    changed = 0
    for b in range(1, total // batch_size + 1):
        if b in done:
            continue
        lo, hi = (b - 1) * batch_size + 1, b * batch_size
        spark.sql(f"""
          MERGE INTO delta.`{q10}` t
          USING (SELECT id, CASE WHEN value < 20 THEN 'low' WHEN value < 35 THEN 'mid' ELSE 'high' END AS nb
                 FROM delta.`{q10}` WHERE id BETWEEN {lo} AND {hi}) s
          ON t.id = s.id
          WHEN MATCHED AND t.value_bucket IS NULL THEN UPDATE SET t.value_bucket = s.nb
        """)
        spark.createDataFrame([(b,)], "batch_id int").write.mode("append").format("delta").save(ledger)
        changed += 1
    return changed

first, second = run_backfill(), run_backfill()
nulls = spark.sql(f"SELECT count(*) FROM delta.`{q10}` WHERE value_bucket IS NULL").collect()[0][0]
buckets = spark.sql(f"SELECT value_bucket, count(*) c FROM delta.`{q10}` GROUP BY 1 ORDER BY 1").toPandas()
print(f"batches: first={first}, re-run={second}, nulls left={nulls}")
print(buckets.to_string(index=False))
assert first == 5 and second == 0 and nulls == 0
assert spark.sql(f"SELECT value_bucket FROM delta.`{q10}` WHERE id = 40").collect()[0][0] == "high"
assert buckets[buckets.value_bucket == "low"].c.iloc[0] == 19
print("Q10 OK: ALTER metadata-only, 5 ledgered batches, idempotent re-run, zero nulls, buckets correct")
print("\nTalk track:")
print("* UPDATE rewrites affected files entirely — a 50M-row single UPDATE is one giant transaction; batched MERGE scopes blast radius to one batch.")
print("* The ledger makes the job restartable ANYWHERE: on failure, rerun skips done batches — idempotency by construction, not by re-scanning.")
print("* Keep the WHEN MATCHED AND ... IS NULL guard: concurrent writers inserting already-bucketed rows stay untouched.")

## Q11 — ETL — Multi-Source CDC Fan-In with a Conflict Matrix `[FAANG]`

**Scenario (data-integration interview):** Customer data arrives from THREE SaaS systems — Stripe (payments), CRM (sales), Support (tickets) — each with its own clock and update cadence. Build one `dim_customer` from the union.

Requirements:
* Resolution rule: newest `updated_at` wins; on an EXACT timestamp tie, a source-priority matrix decides (support > crm > stripe — support tickets carry the churn signal).
* Implement with `row_number() OVER (PARTITION BY customer_id ORDER BY updated_at DESC, src_priority DESC)` — note Spark rejects referencing a lateral alias (`src_priority`) inside the window of the same SELECT, so compute it in an inner CTE.
* Quantify per-source freshness lag (hours behind the fastest source) — this feeds your SLA dashboard.
* MERGE winners into an existing dim: `WHEN MATCHED AND t.updated_at < s.updated_at THEN UPDATE SET *` plus insert-new.
* Prove the tie case: customer 1 updated at the same instant by stripe and support — support must win.

In [0]:
# Solution Q11 — priority tiebreak + per-source lag + MERGE (validated)
def src_df(rows):
    return spark.createDataFrame(rows, "src string, customer_id int, updated_at timestamp, name string, status string")

stripe = src_df([("stripe", 1, dt.datetime(2026, 11, 20, 10), "Ann", "active"),
                 ("stripe", 2, dt.datetime(2026, 11, 20, 11), "Bob", "active")])
crm    = src_df([("crm", 1, dt.datetime(2026, 11, 19, 9), "Ann Lee", "active"),
                 ("crm", 3, dt.datetime(2026, 11, 20, 8), "Cara", "trial")])
sup    = src_df([("support", 1, dt.datetime(2026, 11, 20, 10), "Ann from support", "active"),
                 ("support", 4, dt.datetime(2026, 11, 18, 10), "Dee", "churned")])

united = stripe.unionByName(crm).unionByName(sup)
united.createOrReplaceTempView("cdc_unioned")

dim = spark.sql("""
WITH base AS (
  SELECT *, CASE src WHEN 'support' THEN 3 WHEN 'crm' THEN 2 ELSE 1 END AS src_priority
  FROM cdc_unioned
),
scored AS (
  SELECT *, row_number() OVER (PARTITION BY customer_id
                               ORDER BY updated_at DESC, src_priority DESC) AS rn
  FROM base
)
SELECT src, customer_id, updated_at, name, status FROM scored WHERE rn = 1
""")
dim_pd = dim.orderBy("customer_id").toPandas()
print(dim_pd.to_string(index=False))
assert len(dim_pd) == 4
assert dim_pd[dim_pd.customer_id == 1].src.iloc[0] == "support"      # exact ts tie -> priority wins
assert dim_pd[dim_pd.customer_id == 1].name.iloc[0] == "Ann from support"
assert dim_pd[dim_pd.customer_id == 2].src.iloc[0] == "stripe"         # sole source
assert dim_pd[dim_pd.customer_id == 3].src.iloc[0] == "crm"
assert dim_pd[dim_pd.customer_id == 4].src.iloc[0] == "support"

# per-source freshness lag
newest = spark.sql("SELECT src, max(updated_at) AS newest FROM cdc_unioned GROUP BY src ORDER BY src").toPandas()
newest["lag_hours_vs_fastest"] = (newest.newest.max() - newest.newest).dt.total_seconds() / 3600
print(newest.to_string(index=False))
assert set(newest.src) == {"crm", "stripe", "support"}
assert 0 < newest.lag_hours_vs_fastest.max() < 48

# MERGE into the dim
q11 = f"{BASE}_q11_dim_customer"
(spark.createDataFrame([(1, "old Ann", "unknown", dt.datetime(2026, 1, 1))],
  "customer_id int, name string, status string, updated_at timestamp")
 .write.mode("overwrite").format("delta").save(q11))
dim.createOrReplaceTempView("winners")
spark.sql(f"""
  MERGE INTO delta.`{q11}` t
  USING winners s ON t.customer_id = s.customer_id
  WHEN MATCHED AND t.updated_at < s.updated_at THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
""")
final = spark.sql(f"SELECT customer_id, name, status FROM delta.`{q11}` ORDER BY customer_id").toPandas()
print(final.to_string(index=False))
assert len(final) == 4 and final[final.customer_id == 1].name.iloc[0] == "Ann from support"
print("Q11 OK: support-priority tiebreak on exact ties, sole-source passthrough, per-source lag quantified, MERGE upserts all 4")
print("\nTalk track:")
print("* Ties are REAL: batch CDC exports round timestamps; without the priority tiebreak row_number is nondeterministic across runs.")
print("* Per-source lag = newest-vs-fastest — the number that drives 'source is stale' alerts long before data is wrong.")
print("* At scale, pre-filter each source by its own high-watermark before the union — don't shuffle all history every micro-batch.")

## Q12 — Data Modeling — Junk Dimension: Collapse Low-Cardinality Flags `[HARD]`

**Scenario (Kimball-at-scale interview):** An orders fact carries 7 low-cardinality attributes — 3 booleans, a channel, a segment, an opt-in flag. That's 7 foreig keys and 7 joins on every query; naive flag-dimension modeling would explode into 2^7 = 128 dimension rows *per combination that actually occurs*. The Kimball answer for unrelated low-cardinality attributes is a **junk dimension** — ONE dimension holding every observed combination.

Requirements:
* Build the junk dim: `md5(concat_ws('|', ...))` over the DISTINCT observed combos + a usage count; 120 sample orders across 6 combos.
* Prove the fact shrinks to `(order_id, junk_key, revenue)` and flags are recoverable by joining back — exactly 120 roundtrip rows.
* Storage math: wide 7-column fact vs 32-char junk_key + dim rows.
* New unseen combo arrives (kiosk/vip): show the incremental MERGE payload (distinct new combos) that keeps the dim in sync — re-runnable.

In [0]:
# Solution Q12 — junk dimension build + roundtrip + incremental merge payload (validated)
random.seed(31)
combos = [
    (1, 1, 0, "web", "new", 0), (0, 1, 1, "web", "returning", 0), (1, 0, 0, "mobile", "new", 1),
    (0, 0, 1, "mobile", "returning", 0), (1, 1, 1, "web", "vip", 1), (0, 0, 0, "kiosk", "new", 0),
]
flag_rows = [(i, *combos[i % len(combos)], float(i * 5)) for i in range(120)]
facts = spark.createDataFrame(flag_rows,
    "order_id int, is_new int, is_returning int, is_loyalty int, channel string, segment string, opted_in int, revenue double")
facts.createOrReplaceTempView("facts_raw")

spark.sql("""
  CREATE OR REPLACE TEMP VIEW junk_dim AS
  SELECT md5(concat_ws('|', is_new, is_returning, is_loyalty, channel, segment, opted_in)) AS junk_key,
         is_new, is_returning, is_loyalty, channel, segment, opted_in,
         count(*) AS uses_in_sample
  FROM facts_raw GROUP BY 1, 2, 3, 4, 5, 6, 7
""")
jd = spark.sql("SELECT * FROM junk_dim ORDER BY uses_in_sample DESC").toPandas()
print(jd.to_string(index=False))
assert len(jd) == 6                      # 6 observed combos, NOT 2^7 = 128
assert jd.uses_in_sample.sum() == 120

shrunk = spark.sql("""
  SELECT f.order_id, j.junk_key, f.revenue
  FROM facts_raw f JOIN junk_dim j
    ON j.is_new = f.is_new AND j.is_returning = f.is_returning AND j.is_loyalty = f.is_loyalty
   AND j.channel = f.channel AND j.segment = f.segment AND j.opted_in = f.opted_in
""")
assert shrunk.count() == 120
rt = shrunk.join(spark.sql("SELECT junk_key, channel, segment FROM junk_dim"), "junk_key")
rtp = rt.orderBy("order_id").limit(3).toPandas()
assert list(rtp.order_id) == [0, 1, 2] and rtp[rtp.order_id == 0].channel.iloc[0] == "web"

wide_bytes = 120 * (1 + 1 + 1 + 6 + 10 + 1)
junk_bytes = 120 * 32 + 6 * 20
print(f"storage: wide ~{wide_bytes}B vs junk-keyed ~{junk_bytes}B -> {wide_bytes/junk_bytes:.1f}x fact-row shrink + ONE join instead of seven")

# incremental sync: new unseen combo -> the MERGE payload
new_facts = spark.createDataFrame([(999, 0, 1, 0, "kiosk", "vip", 1, 9.0)],
    "order_id int, is_new int, is_returning int, is_loyalty int, channel string, segment string, opted_in int, revenue double")
new_facts.createOrReplaceTempView("facts_raw2")
spark.sql("CREATE OR REPLACE TEMP VIEW new_combos AS SELECT is_new, is_returning, is_loyalty, channel, segment, opted_in, count(*) uses_in_sample FROM facts_raw2 GROUP BY ALL")
nc = spark.sql("SELECT * FROM new_combos").toPandas()
assert len(nc) == 1 and nc.channel.iloc[0] == "kiosk" and nc.segment.iloc[0] == "vip"
print("new combo payload:", nc.to_dict("records")[0])
print("Q12 OK: 6 combos from 120 rows, join roundtrip exact, new-combo MERGE payload proven")
print("\nTalk track:")
print("* Junk dims are for UNRELATED low-cardinality attrs. Related flags belong in the same combo only if they co-vary; otherwise you still win because observed combos are sparse.")
print("* md5 of the pipe-joined combo = deterministic junk_key -> idempotent MERGE into the dim (ON junk_key, WHEN NOT MATCHED INSERT).")
print("* Watch growth: if combos explode (say 50+), you modeled correlated attrs — split into two junk dims or promote a real dimension.")

## Q13 — Data Modeling — LLM Telemetry Star Schema (GenAI Observability) `[FAANG]`

**Scenario (2025-26 trending — every company shipping LLM features needs this):** Design the dimensional model behind LLM usage analytics: spend per model per day, latency SLAs per endpoint, token economics, cache savings — and the board question "what did the model-upgrade cost us?"

Requirements:
* Grain: ONE ROW PER REQUEST (`request_id`, ts, user, model_version, endpoint, prompt/completion tokens, latency, cost, cache_hit).
* Dims: `dim_model_version` (SCD-2 — model deprecations must be time-traceable), `dim_endpoint`, `dim_user` (consider degenerate key if millions of users), `dim_date`.
* Distinguish **additive** measures (cost_usd, tokens — sum away) from **non-additive/ratio** measures (avg tokens/request, cost per 1K tokens — must be recomputed from sums, never averaged across days).
* Demo: cost per model, avg tokens/request per model — assert the correct values on a 4-row fixture.
* Talk track: why model_version must be SCD-2 (blame the upgrade, not the traffic), cardinality guidance, and the cache-hit degenerate flag.

In [0]:
# Solution Q13 — fixture + additive vs ratio measures (validated)
llm_fact = spark.createDataFrame([
    ("r1", dt.datetime(2026, 11, 20, 9, 1), 101, "gpt-4o-mini", "chat-prod", 120, 480, 950.0, 0.0021, True),
    ("r2", dt.datetime(2026, 11, 20, 9, 2), 102, "gpt-4o-mini", "chat-prod", 80, 120, 610.0, 0.0008, True),
    ("r3", dt.datetime(2026, 11, 20, 9, 3), 101, "llama-3.1-70b", "chat-prod", 300, 1500, 2200.0, 0.0095, False),
    ("r4", dt.datetime(2026, 11, 20, 9, 4), 103, "llama-3.1-70b", "summarize-batch", 900, 900, 8800.0, 0.0052, False),
], "request_id string, ts timestamp, user_id int, model_version string, endpoint string, prompt_tokens int, completion_tokens int, latency_ms double, cost_usd double, cache_hit boolean")
llm_fact.createOrReplaceTempView("fct_llm_requests")

q13a = spark.sql("""
  SELECT model_version,
         round(sum(cost_usd), 6) AS total_cost,               -- additive
         round(avg(latency_ms), 1) AS avg_latency,
         round(sum(prompt_tokens + completion_tokens) / 1e6, 3) AS total_tokens_m,
         round(avg(prompt_tokens + completion_tokens), 1) AS avg_tokens_per_req   -- NON-additive ratio
  FROM fct_llm_requests GROUP BY model_version ORDER BY model_version
""").toPandas()
print(q13a.to_string(index=False))
assert abs(q13a[q13a.model_version == "gpt-4o-mini"].total_cost.iloc[0] - 0.0029) < 1e-9
assert q13a[q13a.model_version == "gpt-4o-mini"].avg_tokens_per_req.iloc[0] == 400.0
assert abs(q13a[q13a.model_version == "llama-3.1-70b"].avg_tokens_per_req.iloc[0] - 1800.0) < 1
print("Q13 OK: additive cost vs ratio tokens-per-request computed correctly on the fixture")
print("\nTalk track:")
print("* DDL sketch: dim_model_version(model_key, model_version, provider, context_window, effective_from, effective_to, is_current) — SCD-2 MERGE on version + is_current flag.")
print("* The wrong-aggregation bug: 'avg tokens/request' per DAY averaged across days = Simpson's paradox; always recompute ratio = sum(tokens)/count(requests).")
print("* dim_user at millions of rows -> keep user_id as degenerate dimension in the fact; don't build a join nobody can afford.")
print("* cache_hit as degenerate flag in-fact; cache savings = sum(cost where cache_hit) x discount — a single additive rollup.")
print("* model deprecation answers: cost by model_version across time, join facts at REQUEST ts (PIT) so the deprecated model's spend is attributable.")

## Q14 — Snowflake — No Row Locks: Snapshot Isolation + OCC Transaction Model `[HARD]`

**Scenario (senior Snowflake interview):** Two analysts run `UPDATE` on the same table concurrently. Postgres would block or deadlock on row locks. Snowflake does NEITHER — there are no row-level locks at all. Explain how concurrent transactions still work, and why you still get aborts.

Requirements (talk track + simulation):
* Writers get isolated table **snapshots** (MVCC): each sees a consistent version; readers never block writers and vice versa.
* On COMMIT, Snowflake checks whether the writer's modified **micro-partitions** were changed by a concurrently committed transaction — optimistic concurrency control (OCC). Overlap → one transaction ABORTS ("could not commit due to concurrent update") and must retry; disjoint micro-partitions → both commit.
* Contrast with Delta Lake's WriteSerializable OCC (Day 25) — same family, different granularity.
* Simulate: two writers touching overlapping vs disjoint micro-partition ranges; overlapping → exactly one commits, the loser retries with exponential backoff and succeeds on attempt 2. Reference SQL: Scripting retry wrapper with `BEGIN ... ROLLBACK`.
* Say why deadlock is IMPOSSIBLE under this model and when OCC retry storms become a real risk.

In [0]:
# Solution Q14 — OCC simulation + reference SQL (validated)
print("""
Reference SQL (Snowflake Scripting retry wrapper):

  DECLARE retry INT DEFAULT 0;
  BEGIN
    LOOP
      BEGIN TRANSACTION;
        UPDATE orders SET status = 'shipped' WHERE region = 'APAC';
      COMMIT;                           -- OCC check happens HERE, not at UPDATE time
        RETURN 'committed on attempt ' || (retry + 1);
      EXCEPTION WHEN OTHERS THEN
        ROLLBACK;
        retry := retry + 1;
        IF retry > 3 THEN RAISE; END IF;
        CALL SYSTEM$WAIT(0.1 * POWER(2, retry), 'SECONDS');   -- exponential backoff
    END LOOP;
  END;
""")

# --- simulation: micro-partition overlap decides who commits ---
random.seed(5)
mp = lambda key: key // 10                          # 10 rows per micro-partition

def simulate_two_writers(w1_rows, w2_rows):
    """Both writers read the same snapshot, modify rows, commit in order.
    OCC: if writer 2's micro-partitions overlap any committed by writer 1 -> writer 2 aborts."""
    committed_mps = set(mp(r) for r in w1_rows)      # writer 1 commits first
    overlap = bool(set(mp(r) for r in w2_rows) & committed_mps)
    w2_commit = not overlap                          # OCC check at commit time
    return w2_commit

# case A: same micro-partition (rows 12, 15) -> writer 2 aborts
assert simulate_two_writers([12, 13], [15, 16]) is False
# case B: disjoint micro-partitions (rows 2 vs rows 55) -> both commit
assert simulate_two_writers([2, 3], [55, 56]) is True

# retry-with-backoff loop: loser succeeds on attempt 2
def occ_retry(attempts_until_clear):
    attempt, waited = 0, 0.0
    while True:
        attempt += 1
        if attempt >= attempts_until_clear:
            return attempt, waited
        waited += 0.1 * (2 ** (attempt - 1))         # exponential backoff

attempts, waited = occ_retry(2)
print(f"overlapping writer aborted once, retried after {waited:.1f}s backoff -> committed on attempt {attempts}")
assert attempts == 2
print("Q14 OK: overlap aborts exactly the second writer; disjoint commits both; retry succeeds at attempt 2")
print("\nTalk track:")
print("* Snapshot isolation: every statement sees a version as of its start — readers never lock, writers never wait on readers. No row locks exist to take.")
print("* OCC failure is deferred to COMMIT: the loser of a micro-partition race gets 'could not commit due to concurrent update' and MUST retry — wrap updates in a Scripting retry loop with backoff.")
print("* Deadlock is impossible: nobody holds locks waiting on anybody — but hot micro-partitions (one region updated by many sessions) create retry storms; remedy: partition-wise writes, stagger schedules, or smaller batch scopes.")
print("* Delta Lake WriteSerializable is the same idea at file level — Day 25 covered it; Snowflake's granularity is micro-partitions (16MB-ish) vs Delta's files.")

## Q15 — Snowflake — COPY INTO Error Handling: ABORT vs CONTINUE vs SKIP_FILE `[FAANG]`

**Scenario (pipeline-interview):** Nightly `COPY INTO` from a stage of 100 files lands 3 files with malformed rows. Choose the error policy deliberately: `ON_ERROR` accepts `ABORT_STATEMENT` (default), `CONTINUE` (load valid rows from bad files), `SKIP_FILE`, and `SKIP_FILE_<n>` (skip only if >n errors). Get this wrong and you either load nothing nightly or silently poison the table.

Requirements (talk track + policy simulation):
* Explain each mode + `VALIDATION_MODE = RETURN_ERRORS` (dry-run: parse and report, load nothing) + `COPY_HISTORY`/`result_scan` reconciliation.
* Simulate 100 files where 3 are corrupt: compute loaded files/rows for (a) ABORT_STATEMENT, (b) CONTINUE, (c) SKIP_FILE; assert each policy's outcome.
* Dry-run math: with VALIDATION_MODE first, how many rows does the dry run process and what does it cost (parse-only, no load)?
* Choose-and-justify: third-party vendor drops (SKIP_FILE + quarantine), internal validated producer (ABORT), exploratory (CONTINUE).

In [0]:
# Solution Q15 — policy simulation + reference SQL (validated)
print("""
Reference SQL:

  -- dry run: parse all files, report errors, load NOTHING
  COPY INTO raw_events FROM @stage/path
    FILE_FORMAT = (TYPE = CSV, TRIM_SPACE = TRUE)
    VALIDATION_MODE = RETURN_ERRORS;                -- returns per-file error rows via result_scan

  -- production load: skip corrupt files, keep the rest, quarantine later via COPY_HISTORY
  COPY INTO raw_events FROM @stage/path
    FILE_FORMAT = (TYPE = CSV)
    ON_ERROR = SKIP_FILE                            -- or SKIP_FILE_10; ABORT_STATEMENT is default
    PURGE = FALSE;                                  -- keep files for the quarantine step

  -- reconciliation: what did the load actually do?
  SELECT file_name, row_count, row_parsed, errors_seen, first_error
  FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(TABLE_NAME => 'RAW_EVENTS', START_TIME => DATEADD(hour, -24, CURRENT_TIMESTAMP())));
""")

# --- policy simulation: 100 files, 3 corrupt (1 bad row each), 1000 rows/file ---
N_FILES, CORRUPT, ROWS = 100, 3, 1000

def policy_outcome(on_error):
    if on_error == "ABORT_STATEMENT":
        return 0, 0                      # whole statement rolls back — nothing loaded
    if on_error == "CONTINUE":
        return N_FILES, N_FILES * ROWS - CORRUPT      # bad ROWS dropped, bad files' good rows load
    if on_error == "SKIP_FILE":
        return N_FILES - CORRUPT, (N_FILES - CORRUPT) * ROWS  # corrupt files skipped entirely

for mode, want in (("ABORT_STATEMENT", (0, 0)),
                   ("CONTINUE", (N_FILES, N_FILES * ROWS - CORRUPT)),
                   ("SKIP_FILE", (N_FILES - CORRUPT, (N_FILES - CORRUPT) * ROWS))):
    got = policy_outcome(mode)
    print(f"ON_ERROR = {mode:<16} -> loaded files={got[0]:>3}, loaded rows={got[1]:>7}  expected {want}")
    assert got == want

# SKIP_FILE_10 nuance: skip only files with MORE than n errors
print("SKIP_FILE_10: files with 1 bad row still LOAD (minus that row) — only files with >10 errors get skipped")
dry_run_rows = N_FILES * ROWS
print(f"VALIDATION_MODE=RETURN_ERRORS parses all {dry_run_rows:,} rows but loads 0 — parse-only cost, ~same CPU as the load, half the I/O")
print("Q15 OK: all three policies simulated exactly; dry-run and reconciliation paths referenced")
print("\nTalk track:")
print("* Default ABORT fails the whole statement on the FIRST bad file — a 3-file vendor error blocks 97 good files unless you pre-validate.")
print("* CONTINUE maximizes ingestion but writes partial data from corrupt files — fine for exploratory, risky for downstream contracts.")
print("* Production pattern: VALIDATION_MODE dry-run -> quarantine stage for files with errors -> COPY with ON_ERROR=SKIP_FILE + PURGE=FALSE -> COPY_HISTORY reconciliation alert.")
print("* Vendor drops: SKIP_FILE + alert per skipped file; internal validated producers: keep ABORT (fail loudly = your contract is broken).")

## Q16 — Monitoring & QA — Synthetic Test Data with Statistical Fidelity `[FAANG]`

**Scenario (data-testing interview):** Your pipeline tests run on 2,000-row fixtures. Every time a production incident happens (revenue distribution shift, channel-mix change), the tests never caught it. The fixtures were uniform — they matched nothing real. Build a synthetic-data generator that PRESERVES production's statistical shape.

Requirements:
* Fixture from a production-like sample: skewed categorical channel (web 55%, ios 25%, android 15%, desktop 5%) + gamma-shaped revenue + user IDs constrained to valid range (referential integrity).
* Implement two fidelity scores: total-variation distance for categoricals and Kolmogorov–Smirnov distance for numerics.
* Naive fixture (uniform categories + wrong-mean normal revenue) must FAIL both; your generator (empirical-marginal sampling + jittered bootstrap for revenue) must PASS at TV < 0.05 and KS < 0.05.
* Prove FK integrity: every synthetic user_id references a valid user.
* Talk track: what these fixtures catch that snapshots miss (drift), and when you must NOT sample production data (PII).

In [0]:
# Solution Q16 — TV + KS fidelity scoring (validated: naive TV 0.299/KS 0.637 rejected; marginals 0.012/0.017 pass)
rng = np.random.default_rng(42)

prod_channels = rng.choice(["web", "ios", "android", "desktop"], size=5000, p=[0.55, 0.25, 0.15, 0.05])
prod_revenue = rng.gamma(shape=2.0, scale=25.0, size=5000)

def tv_distance(a_counts, b_counts):
    allk = sorted(set(a_counts) | set(b_counts))
    na, nb = sum(a_counts.values()), sum(b_counts.values())
    return 0.5 * sum(abs(a_counts.get(k, 0) / na - b_counts.get(k, 0) / nb) for k in allk)

def ks_distance(x, y):
    xs, ys = np.sort(x), np.sort(y)
    allv = np.concatenate([xs, ys])
    cdf_x = np.searchsorted(xs, allv, side="right") / len(xs)
    cdf_y = np.searchsorted(ys, allv, side="right") / len(ys)
    return float(np.max(np.abs(cdf_x - cdf_y)))

# (a) NAIVE: uniform categories + wrong-mean normal revenue
bad_ch = rng.choice(["web", "ios", "android", "desktop"], size=2000)
bad_rev = rng.normal(20.0, 5.0, size=2000)

# (b) FIDELITY: empirical marginals + jittered bootstrap revenue
chan_props = [float((prod_channels == c).mean()) for c in ["web", "ios", "android", "desktop"]]
good_ch = rng.choice(["web", "ios", "android", "desktop"], size=2000, p=chan_props)
good_rev = rng.choice(prod_revenue, size=2000, replace=True) + rng.normal(0, 0.5, 2000)
good_users = rng.integers(1, 401, size=2000)

cnt = Counter
tv_bad = tv_distance(cnt(prod_channels), cnt(bad_ch))
tv_good = tv_distance(cnt(prod_channels), cnt(good_ch))
ks_bad, ks_good = ks_distance(prod_revenue, bad_rev), ks_distance(prod_revenue, good_rev)
print(f"categorical TV:  uniform={tv_bad:.4f}  marginal-fit={tv_good:.4f}")
print(f"revenue KS:      wrong-normal={ks_bad:.4f}  jittered-bootstrap={ks_good:.4f}")
assert tv_bad > 0.25 and tv_good < 0.05
assert ks_bad > 0.4 and ks_good < 0.05
assert good_users.min() >= 1 and good_users.max() <= 400          # FK integrity preserved
print("Q16 OK: naive fixture rejected (TV>0.25, KS>0.4); fidelity fixture passes (TV<0.05, KS<0.05); user FK integrity kept")
print("\nTalk track:")
print("* TV distance: half the L1 distance between categorical distributions; 0 = identical mix, ~0.75 = uniform vs heavy skew.")
print("* KS distance: max gap between empirical CDFs — catches shape (gamma vs normal) even when means match.")
print("* Correlations: preserve pairwise shape by conditional sampling or copulas; marginals alone miss joint drift.")
print("* PII rule: fit distributions on production, SAMPLE in the test env from anonymized marginals — never copy rows with identifiers.")

## Q17 — Monitoring & QA — Consumer-Query Replay Promotion Gate `[HARD]`

**Scenario (data-platform interview):** You're promoting v2 of the `orders` table (schema additions, upstream 'fixes'). Dashboards downstream passed — but dashboards only cover 20% of consumer queries. A silent semantics change (e.g., refunds no longer subtracted from a column) would ship. Build a promotion gate on **consumer-query replay**.

Requirements:
* Maintain a registry of top consumer queries (from `system.query.history`); each gets a deterministic checksum over its full result (sorted, JSON-serialized, md5).
* Replay all registered queries on the candidate version; a change is EITHER benign (checksum identical — e.g. added column) or a SEMANTICS change (checksum differs).
* Demonstrate on a 4-row Delta table: v2 adds a `loyalty_pts` column (all checksums stable → promote); v3 zeroes the `refunds` column (finance + fraud queries change → block).
* Talk track: checksum pitfalls (order, types, floating point), which queries to register (top-N by frequency × criticality), and how this complements — not replaces — unit tests and data-quality gates.

In [0]:
# Solution Q17 — checksum replay gate (validated: benign add promotes, semantics change blocks)
q17 = f"{BASE}_q17_orders"
orders_v1 = [(1, "A", 100.0, 20.0), (2, "B", 250.0, 20.0), (3, "A", 80.0, 0.0), (4, "C", 400.0, 50.0)]
(spark.createDataFrame(orders_v1, "id int, region string, gross double, refunds double")
 .write.mode("overwrite").format("delta").save(q17))

CONSUMER_QUERIES = {
    "finance_net_revenue": "SELECT region, sum(gross - refunds) net FROM {T} GROUP BY region ORDER BY region",
    "ops_order_count":     "SELECT count(*) c FROM {T} WHERE gross > 50",
    "fraud_refund_rate":   "SELECT round(sum(refunds) / sum(gross), 3) r FROM {T}",
}

def checksum(rows):
    return hashlib.md5(json.dumps(sorted(rows), sort_keys=True, default=str).encode()).hexdigest()

def replay(table_path):
    return {name: checksum(spark.sql(q.format(T=f"delta.`{table_path}`")).toJSON().collect())
            for name, q in CONSUMER_QUERIES.items()}

v1_sums = replay(q17)
print("v1 checksums:", v1_sums)

# v2: harmless schema addition (new column, semantics unchanged)
(spark.read.format("delta").load(q17).withColumn("loyalty_pts", F.col("id") * 10)
 .write.mode("overwrite").option("mergeSchema", "true").format("delta").save(q17))
v2_sums = replay(q17)
diffs2 = {k: v1_sums[k] == v2_sums[k] for k in v1_sums}
print("v2 diff flags (benign schema add):", diffs2)
assert all(diffs2.values()), "benign addition must not change consumer results"

# v3: semantics change — refunds zeroed upstream (an upstream 'fix' breaks finance semantics)
(spark.read.format("delta").load(q17).withColumn("refunds", F.lit(0.0))
 .write.mode("overwrite").format("delta").save(q17))
v3_sums = replay(q17)
diffs3 = {k: v1_sums[k] == v3_sums[k] for k in v1_sums}
print("v3 diff flags (refund semantics change):", diffs3)
assert diffs3["ops_order_count"] is True
assert diffs3["finance_net_revenue"] is False
assert diffs3["fraud_refund_rate"] is False
print("Q17 OK: benign schema add promotes (checksums stable); refund-semantics change caught and blocked")
print("\nTalk track:")
print("* Determinism: sort rows + sort keys + default=str before hashing — unordered results, decimals, timestamps must canonicalize identically.")
print("* Register top-N queries from system.query.history by frequency x business-criticality; rotate stale queries monthly.")
print("* A changed checksum is not automatically 'bad' — it's a REVIEW gate: diff row samples per query, then decide promote/revert/notify owners.")
print("* This complements unit tests (logic), DQ gates (data validity) and shadow deploys (runtime): it protects the CONTRACT between table and consumers.")

## Q18 — Monitoring & QA — Query Performance Regression CI `[FAANG]`

**Scenario (platform-engineering interview):** A teammate's "harmless refactor" of a dashboard query added a Python UDF — dashboard p95 went from 0.7s to 1.2s and nobody noticed for a month. Your CI treats correctness (tests pass) but never **performance**. Build a regression gate for a registered query set.

Requirements:
* Benchmark harness: warmup 2 runs (JIT/codegen + cache — discarded), then median of 5 samples; track MAD for noise.
* Gate: flag a REGRESSION when current median > baseline median × 1.25 + max(3×MAD, 50ms floor) — noise-aware, not naive threshold.
* Demonstrate: (a) fresh run of the same queries must PASS; (b) one query swapped for a per-row Python-UDF rewrite must get FLAGGED.
* Table: 400k rows in Delta; three registered queries (aggregate by device, aggregate by sensor, top-N).
* Talk track: why median+MAD beats mean, warmup discipline, interleaving A/B runs against system noise, and where the baseline JSON lives in CI.

In [0]:
# Solution Q18 — median+MAD perf gate (validated: clean rerun passes; per-row-UDF rewrite flagged)
q18 = f"{BASE}_q18_events"
big = spark.range(1, 400001).select(
    (F.col("id") % 50).alias("device"),
    (F.col("id") % 1000).alias("sensor"),
    (F.col("id").cast("double") / 7).alias("reading"))
big.write.mode("overwrite").format("delta").save(q18)

QUERIES = {
    "agg_by_device": f"SELECT device, count(*), round(avg(reading), 4) FROM delta.`{q18}` GROUP BY device",
    "agg_by_sensor": f"SELECT sensor, count(*) FROM delta.`{q18}` GROUP BY sensor",
    "top_readings":  f"SELECT * FROM delta.`{q18}` ORDER BY reading DESC LIMIT 20",
}

def run_once(sql):
    t0 = time.time()
    spark.sql(sql).collect()
    return time.time() - t0

def benchmark(sql, warmup=2, samples=5):
    for _ in range(warmup): run_once(sql)          # warm cache + codegen — discard
    return [run_once(sql) for _ in range(samples)]

def mad(xs):
    m = statistics.median(xs)
    return statistics.median([abs(x - m) for x in xs])

def collect_perf(queries):
    return {name: {"median": statistics.median(benchmark(sql)), "mad": mad(benchmark(sql))}
            for name, sql in queries.items()}

def gate(baseline, current, factor=1.25, noise_mult=3.0, floor=0.05):
    report, failed = [], []
    for name in baseline:
        b, c = baseline[name]["median"], current[name]["median"]
        spread = max(baseline[name]["mad"] * noise_mult, floor)
        status = "REGRESSION" if c > b * factor + spread else "OK"
        report.append((name, round(b, 3), round(c, 3), status))
        if status == "REGRESSION":
            failed.append(name)
    return report, failed

baseline = collect_perf(QUERIES)
print("baseline:", {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in baseline.items()})

# run 1: same queries, fresh measurement — must pass
report_ok, failed_ok = gate(baseline, collect_perf(QUERIES))
for row in report_ok: print(row)
assert failed_ok == [], failed_ok

# run 2: one query swapped for a per-row Python-UDF rewrite — must flag
def heavy(v):
    v = abs(float(v))
    return sum(math.sin(v + i * 0.1) for i in range(30)) + v
spark.udf.register("heavy_feature", heavy)
REGRESSED_REWRITE = f"SELECT device, count(*), round(avg(heavy_feature(reading)), 4) FROM delta.`{q18}` GROUP BY device"

regressed = dict(QUERIES); regressed["agg_by_device"] = REGRESSED_REWRITE
report_bad, failed_bad = gate(baseline, collect_perf(regressed))
for row in report_bad: print(row)
print("failed queries:", failed_bad)
assert failed_bad == ["agg_by_device"], failed_bad
print("Q18 OK: clean rerun passes gate; per-row-UDF rewrite flagged by median+noise gate")
print("\nTalk track:")
print("* Median over 5 samples resists stragglers; MAD (not std) resists outliers in the noise estimate itself.")
print("* Warmup is non-negotiable: first-touch pages + codegen compile can be 3-10x the steady state.")
print("* In CI: store baseline JSON per table version in the repo; regenerate on approved changes only (like a snapshot test for speed).")
print("* False alarms: shared cluster contention — run on a dedicated small warehouse/cluster; interleave baseline vs candidate runs to cancel drift.")

## Day 30 — Self-Assessment Scorecard

Score yourself 1–5 per question **after** attempting it, before reading the solution.

| # | Topic | Tag | Score (1-5) | Confidence (1-5) | Notes |
|---|---|---|---|---|---|
| 1 | Loan amortization + underwater LTV in recursive SQL | HARD | | | |
| 2 | Sequential Elo ratings via map-state recursive CTE | FAANG | | | |
| 3 | Sparse-table RMQ (O(1) queries) | HARD | | | |
| 4 | Cuckoo hashing (hops, cycles, bounded rehash) | FAANG | | | |
| 5 | Distributed blocked matrix multiply | FAANG | | | |
| 6 | Distributed logistic regression (mapInPandas gradients) | HARD | | | |
| 7 | Barrier execution & gang scheduling | HARD | | | |
| 8 | GPU acceleration with RAPIDS + transfer math | FAANG | | | |
| 9 | Lifecycle tiering, VACUUM retention, cost break-even | FAANG | | | |
| 10 | Incremental column backfill with progress ledger | HARD | | | |
| 11 | Multi-source CDC fan-in + conflict matrix | FAANG | | | |
| 12 | Junk dimension (collapse low-cardinality flags) | HARD | | | |
| 13 | LLM telemetry star schema (additive vs ratio) | FAANG | | | |
| 14 | Snowflake OCC: no row locks, snapshot isolation | HARD | | | |
| 15 | COPY INTO error modes (ABORT/CONTINUE/SKIP_FILE) | FAANG | | | |
| 16 | Synthetic test data with TV/KS fidelity | FAANG | | | |
| 17 | Consumer-query replay promotion gate | HARD | | | |
| 18 | Query performance regression CI | FAANG | | | |

**Day 30 tags: 10 × [FAANG], 8 × [HARD]**

**Red flags to review tonight** (any of these = re-read that solution):

* Used `max(balance_out)` for the final amortization balance, or joined appraisals on `loan_id` alone
* Couldn't say why the sparse table needs idempotent min/max, or why sum-RMQ dies
* Let a (n,1) numpy vector touch an (n,) vector in the gradient UDF — or didn't spot why it's silently n× wrong
* Claimed barrier mode retries only the failed task (it retries the whole stage)
* Forgot PCIe transfer time in the GPU speedup estimate
* Asserted VACUUM reclaims everything immediately (retention window gates it)
* Would let a multi-source CDC tie be resolved by nondeterministic ordering
* Averaged a ratio metric across days (Simpson's paradox) in the LLM telemetry rollup

---

*Day 30 complete — streak: 30/30. On to Day 31.*